In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum

# Create Spark Session
spark = SparkSession.builder \
    .appName("AdvSparkApp") \
    .getOrCreate()

# Create DataFrame with 1,000,000 rows
df = spark.range(1, 1000000).withColumn("value", col("id") % 1000)

# Show first 100 rows
df.show(100)

# Check number of partitions before repartitioning
print("Before Repartition:", df.rdd.getNumPartitions())

# Repartition DataFrame
df_repartitioned = df.repartition(8)

# Check number of partitions after repartitioning
print("After Repartition:", df_repartitioned.rdd.getNumPartitions())

# Coalesce partitions
df_coalesced = df_repartitioned.coalesce(4)

# Check number of partitions after coalescing
print("After Coalesce:", df_coalesced.rdd.getNumPartitions())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 05:44:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---+-----+
| id|value|
+---+-----+
|  1|    1|
|  2|    2|
|  3|    3|
|  4|    4|
|  5|    5|
|  6|    6|
|  7|    7|
|  8|    8|
|  9|    9|
| 10|   10|
| 11|   11|
| 12|   12|
| 13|   13|
| 14|   14|
| 15|   15|
| 16|   16|
| 17|   17|
| 18|   18|
| 19|   19|
| 20|   20|
| 21|   21|
| 22|   22|
| 23|   23|
| 24|   24|
| 25|   25|
| 26|   26|
| 27|   27|
| 28|   28|
| 29|   29|
| 30|   30|
| 31|   31|
| 32|   32|
| 33|   33|
| 34|   34|
| 35|   35|
| 36|   36|
| 37|   37|
| 38|   38|
| 39|   39|
| 40|   40|
| 41|   41|
| 42|   42|
| 43|   43|
| 44|   44|
| 45|   45|
| 46|   46|
| 47|   47|
| 48|   48|
| 49|   49|
| 50|   50|
| 51|   51|
| 52|   52|
| 53|   53|
| 54|   54|
| 55|   55|
| 56|   56|
| 57|   57|
| 58|   58|
| 59|   59|
| 60|   60|
| 61|   61|
| 62|   62|
| 63|   63|
| 64|   64|
| 65|   65|
| 66|   66|
| 67|   67|
| 68|   68|
| 69|   69|
| 70|   70|
| 71|   71|
| 72|   72|
| 73|   73|
| 74|   74|
| 75|   75|
| 76|   76|
| 77|   77|
| 78|   78|
| 79|   79|
| 80|   80|
| 81

[Stage 1:>                                                          (0 + 2) / 2]

After Repartition: 8


[Stage 2:>                                                          (0 + 2) / 2]

After Coalesce: 4


26/06/13 05:45:11 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [2]:
import time

start = time.time()

# Your Spark code
df = spark.range(1, 1000000).withColumn("value", col("id") % 1000)

df_repartitioned = df.repartition(8)
df_coalesced = df_repartitioned.coalesce(4)

# Trigger execution
df_coalesced.count()

end = time.time()

print("Execution Time:", end - start, "seconds")

[Stage 5:=============================>                             (2 + 2) / 4]

Execution Time: 1.3084423542022705 seconds


In [3]:
optimized_df = df.filter((col("id") > 1000) & (col("value") > 500))

In [4]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#14L, (id#14L % 1000) AS value#16L]
+- *(1) Filter ((id#14L > 1000) AND ((id#14L % 1000) > 500))
   +- *(1) Range (1, 1000000, step=1, splits=2)




In [5]:
start_time = time.time()

count_uncached = optimized_df.count()

end_time = time.time()

print(f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds")

1. Optimized Execution | Count: 498501 | Time: 0.3418 seconds


In [6]:
optimized_df.cache()

# Materialize cache
optimized_df.count()

start_time = time.time()
count_cached = optimized_df.count()
end_time = time.time()

print(f"2. Cached Execution | Count: {count_cached} | Time: {round(end_time - start_time, 4)} seconds")

2. Cached Execution | Count: 498501 | Time: 0.2084 seconds


In [8]:
print(optimized_df.storageLevel)

Disk Memory Deserialized 1x Replicated


In [9]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [10]:
optimized_df.toPandas().to_csv(
    "output/employees123.csv",
    index=False
)